# Storing the data in a database (SQLite) and SQL queries from Python

**This Jupyter notebook does the following:**

- Creates the SQLite database `Data/topic1_apartments.db`
- Creates two tables with SQL: `apartments` (one row per apartment) and `municipalities` (one row per municipality)
- Writes the enriched data from `Data/apartments_pre1990_enriched.csv` to the tables
- Runs SQL queries from Python (counts, averages, `GROUP BY`, `JOIN` of the two tables)
- Closes the database connection

The structure follows `Week_08/apartments_database_Python.ipynb`.

**Why SQLite?** SQLite is included in Python (module `sqlite3`) and runs in the course environment without a database server. A MySQL or PostgreSQL database would need an extra Python package (e.g. `psycopg2`), which is not part of the course environment.

## Libraries and settings

In [1]:
# Libraries
import os
import sqlite3
import numpy as np
import pandas as pd

# Ignore warnings
import warnings
warnings.filterwarnings("ignore")

# Show current working directory
print(os.getcwd())

C:\Users\DELL\Documents\applied_research_methods\Project_Topic1


### Function to check whether the database connection is open

In [2]:
def check_conn(conn):
    try:
        conn.cursor()
        return True
    except Exception:
        return False

## Create the database and the tables

In [3]:
# Create (or open) the database
conn = sqlite3.connect('./Data/topic1_apartments.db')
cursor = conn.cursor()
print('Connection open:', check_conn(conn))

# Delete old tables, so that the notebook can be run several times
cursor.execute('DROP TABLE IF EXISTS apartments')
cursor.execute('DROP TABLE IF EXISTS municipalities')
conn.commit()

Connection open: True


### Table `municipalities`

One row per municipality. `bfs_number` is the primary key.

In [4]:
cursor.execute('''CREATE TABLE municipalities (
                      bfs_number            INTEGER PRIMARY KEY,
                      bfs_name              VARCHAR(100),
                      canton                VARCHAR(2),
                      language_region       VARCHAR(10),
                      population                   INTEGER,
                      population_density              DECIMAL(10,2),
                      foreigners_pct               DECIMAL(5,2),
                      employees                   INTEGER,
                      mean_taxable_income   DECIMAL(10,2))''')
conn.commit()

### Table `apartments`

One row per apartment. `bfs_number` is a foreign key to the table `municipalities`. True/False columns are stored as `INTEGER` (1/0), because SQLite has no boolean data type.

In [5]:
cursor.execute('''CREATE TABLE apartments (
                      apartment_id          INTEGER PRIMARY KEY,
                      source                VARCHAR(20),
                      listing_url           VARCHAR(300),
                      address               VARCHAR(200),
                      zip                   INTEGER,
                      locality              VARCHAR(100),
                      bfs_number            INTEGER REFERENCES municipalities(bfs_number),
                      lat                   DECIMAL(9,6),
                      lon                   DECIMAL(9,6),
                      rooms                 DECIMAL(4,1),
                      area                  DECIMAL(6,1),
                      price                 DECIMAL(8,2),
                      price_net             DECIMAL(8,2),
                      rent_per_m2           DECIMAL(6,2),
                      rent_net_per_m2       DECIMAL(6,2),
                      year_built            INTEGER,
                      year_renovated        INTEGER,
                      building_age          INTEGER,
                      furnished             INTEGER,
                      temporary             INTEGER,
                      renovation_group      VARCHAR(30),
                      reno_complete         INTEGER,
                      reno_kitchen          INTEGER,
                      reno_bathroom         INTEGER,
                      reno_floors           INTEGER,
                      reno_paint            INTEGER,
                      energy_windows        INTEGER,
                      energy_heating        INTEGER,
                      energy_insulation     INTEGER,
                      energy_roof           INTEGER,
                      energy_solar          INTEGER,
                      energy_minergie       INTEGER,
                      renovated_parts       VARCHAR(200))''')
conn.commit()

# Show the tables in the database
pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table'", con=conn)

,name
0,municipalities
1,apartments


## Write the data to the tables

In [6]:
df = pd.read_csv('./Data/apartments_pre1990_enriched.csv', sep=',', encoding='utf-8')
print('Dimension:', df.shape)

# Table municipalities: one row per municipality of the apartments
df_mun = (df[['bfs_number', 'bfs_name', 'canton', 'language_region', 'population', 'population_density',
              'foreigners_pct', 'employees', 'mean_taxable_income']]
          .drop_duplicates(subset='bfs_number')
          .sort_values('bfs_number'))
df_mun.to_sql('municipalities', con=conn, if_exists='append', index=False)

# Table apartments
bool_cols = ['furnished', 'temporary'] + [c for c in df.columns if c.startswith('reno_') or c.startswith('energy_')]
df_apt = df.rename(columns={'area_clean': 'area',
                            'year_built_clean': 'year_built',
                            'year_renovated_clean': 'year_renovated'})
df_apt[bool_cols] = df_apt[bool_cols].astype(int)
df_apt.insert(0, 'apartment_id', range(1, len(df_apt) + 1))

apt_cols = [r[1] for r in cursor.execute('PRAGMA table_info(apartments)').fetchall()]
df_apt[apt_cols].to_sql('apartments', con=conn, if_exists='append', index=False)

print('Rows in municipalities:', cursor.execute('SELECT COUNT(*) FROM municipalities').fetchone()[0])
print('Rows in apartments:', cursor.execute('SELECT COUNT(*) FROM apartments').fetchone()[0])

Dimension: (726, 50)
Rows in municipalities: 332
Rows in apartments: 726


## SQL queries from Python

### Query 1: first rows of the table `apartments`

In [7]:
pd.read_sql_query('''SELECT apartment_id, source, locality, rooms, area, price, rent_net_per_m2, renovation_group
                     FROM apartments
                     LIMIT 5''', con=conn)

,apartment_id,source,locality,rooms,area,price,rent_net_per_m2,renovation_group
0,1,immoscout24,La Chaux-de-Fonds,4.5,83,1485,14.46,general renovation
1,2,immoscout24,Luterbach,4.5,108,1990,15.09,general renovation
2,3,immoscout24,Lenzburg,4.5,80,1760,18.38,energy renovation
3,4,immoscout24,Uster,5.5,116,2220,16.03,not renovated
4,5,immoscout24,Fribourg,2.0,52,1275,21.73,general renovation


### Query 2: number of apartments and mean rents per renovation group (`GROUP BY`)

In [8]:
pd.read_sql_query('''SELECT renovation_group,
                            COUNT(*)                       AS n,
                            ROUND(AVG(area), 1)            AS mean_area,
                            ROUND(AVG(price), 0)           AS mean_gross_rent,
                            ROUND(AVG(rent_per_m2), 2)     AS mean_gross_rent_m2,
                            ROUND(AVG(rent_net_per_m2), 2) AS mean_net_rent_m2,
                            COUNT(rent_net_per_m2)         AS n_net_rent
                     FROM apartments
                     GROUP BY renovation_group
                     ORDER BY mean_net_rent_m2 DESC''', con=conn)

,renovation_group,n,mean_area,mean_gross_rent,mean_gross_rent_m2,mean_net_rent_m2,n_net_rent
0,energy renovation,43,86.4,2091.0,24.68,23.37,30
1,general renovation,486,87.0,1997.0,23.97,21.09,350
2,not renovated,197,85.3,1882.0,23.55,20.24,174


### Query 3: what was renovated in the energy group? (`SUM` of 1/0 columns)

In [9]:
pd.read_sql_query('''SELECT SUM(energy_heating)    AS heating,
                            SUM(energy_windows)    AS windows,
                            SUM(energy_solar)      AS solar,
                            SUM(energy_insulation) AS insulation,
                            SUM(energy_roof)       AS roof,
                            SUM(energy_minergie)   AS minergie,
                            SUM(reno_kitchen)      AS kitchen,
                            SUM(reno_bathroom)     AS bathroom,
                            COUNT(*)               AS n
                     FROM apartments
                     WHERE renovation_group = 'energy renovation' ''', con=conn)

,heating,windows,solar,insulation,roof,minergie,kitchen,bathroom,n
0,17,13,12,8,2,6,10,10,43


### Query 4: join of the two tables — mean net rent per m² by language region (`JOIN`)

In [10]:
pd.read_sql_query('''SELECT m.language_region,
                            a.renovation_group,
                            COUNT(*)                         AS n,
                            ROUND(AVG(a.rent_net_per_m2), 2) AS mean_net_rent_m2
                     FROM apartments AS a
                     JOIN municipalities AS m ON a.bfs_number = m.bfs_number
                     GROUP BY m.language_region, a.renovation_group
                     ORDER BY m.language_region, a.renovation_group''', con=conn)

,language_region,renovation_group,n,mean_net_rent_m2
0,French,energy renovation,6,22.30
1,French,general renovation,83,23.09
2,French,not renovated,58,24.06
3,German,energy renovation,34,24.20
4,German,general renovation,377,20.83
5,German,not renovated,129,18.74
6,Italian,energy renovation,3,19.09
7,Italian,general renovation,26,18.97
8,Italian,not renovated,10,17.02


### Query 5: cantons with the most energy-renovated apartments (`JOIN`, `WHERE`, `HAVING`)

In [11]:
pd.read_sql_query('''SELECT m.canton,
                            COUNT(*) AS n_apartments,
                            SUM(CASE WHEN a.renovation_group = 'energy renovation' THEN 1 ELSE 0 END) AS n_energy,
                            ROUND(AVG(m.mean_taxable_income), 0) AS mean_taxable_income
                     FROM apartments AS a
                     JOIN municipalities AS m ON a.bfs_number = m.bfs_number
                     GROUP BY m.canton
                     HAVING n_apartments >= 20
                     ORDER BY n_energy DESC''', con=conn)

,canton,n_apartments,n_energy,mean_taxable_income
0,BE,132,11,65651.0
1,ZH,55,5,87711.0
2,SG,67,4,63545.0
3,GR,26,4,66074.0
4,AG,45,4,71855.0
5,TI,39,3,67910.0
6,VS,25,1,59430.0
7,TG,30,1,66682.0
8,SO,34,1,66310.0
9,FR,36,1,67509.0


### Query 6: the 10 municipalities with the highest mean taxable income in the data

In [12]:
pd.read_sql_query('''SELECT m.bfs_name, m.canton, m.mean_taxable_income, COUNT(a.apartment_id) AS n_apartments
                     FROM municipalities AS m
                     JOIN apartments AS a ON a.bfs_number = m.bfs_number
                     WHERE m.mean_taxable_income IS NOT NULL
                     GROUP BY m.bfs_number
                     ORDER BY m.mean_taxable_income DESC
                     LIMIT 10''', con=conn)

,bfs_name,canton,mean_taxable_income,n_apartments
0,Feusisberg,SZ,229181.911653,2
1,Herrliberg,ZH,174166.312178,2
2,Kilchberg (ZH),ZH,166467.202970,2
3,Zollikon,ZH,164756.669310,1
4,Zug,ZG,148339.909534,1
5,Meilen,ZH,130712.349118,1
6,Crassier,VD,127229.613734,1
7,Biel-Benken,BL,124979.194631,1
8,Bottmingen,BL,119781.959263,2
9,Pully,VD,117982.667953,1


### Read the data for the analysis from the database (used in the next notebooks)

In [13]:
df_db = pd.read_sql_query('''SELECT a.*, m.bfs_name, m.canton, m.language_region, m.population, m.population_density,
                                    m.foreigners_pct, m.employees, m.mean_taxable_income
                             FROM apartments AS a
                             JOIN municipalities AS m ON a.bfs_number = m.bfs_number''', con=conn)
print('Dimension:', df_db.shape)

Dimension: (726, 41)


## Close db connection (if open)

In [14]:
if check_conn(conn):
    conn.close()
print('Connection open:', check_conn(conn))

Connection open: False


### Jupyter notebook --footer info-- (please always provide this at the end of each notebook)

In [15]:
import os
import platform
import socket
from platform import python_version
from datetime import datetime

print('-----------------------------------')
print(os.name.upper())
print(platform.system(), '|', platform.release())
print('Datetime:', datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print('Python Version:', python_version())
print('-----------------------------------')

-----------------------------------
NT
Windows | 10
Datetime: 2026-11-26 15:45:45
Python Version: 3.11.8
-----------------------------------
